# 非hot100外常考非MLDL代码题

## 快速排序&归并排序

In [25]:
def quick_sort(nums):
    if len(nums)<=1:
        return nums
    a = nums[len(nums)//2]  
    small = [x for x in nums if x<a]
    mid = [x for x in nums if x==a]
    big = [x for x in nums if x>a]
    return quick_sort(small)+mid+quick_sort(big)
arr = [5, 3, 8, 4, 2, 7, 1, 10]
sorted_arr = quick_sort(arr)
print(sorted_arr)
def sort(l,r):
    res = []
    i,j = 0,0
    while i < len(l) and j < len(r):
        if l[i] < r[j]:
            res.append(l[i])
            i += 1
        else:
            res.append(r[j])
            j += 1
    res.extend(l[i:])
    res.extend(r[j:])
    return res
def merge_sort(nums):
    if len(nums)<=1:
        return nums
    mid = len(nums)//2
    l = merge_sort(nums[:mid])
    r = merge_sort(nums[mid:])
    return sort(l,r)
arr = [5, 3, 8, 4, 2, 7, 1, 10]
sorted_arr = merge_sort(arr)
print(sorted_arr)

[1, 2, 3, 4, 5, 7, 8, 10]
[1, 2, 3, 4, 5, 7, 8, 10]


## 写一个最普通的NNs，搭全流程前向传播后向传播，不用RELU，不用softmax，用torch实现。

In [ ]:
import torch
class SimpleNN:
    """
    最基础的全连接神经网络（无激活函数、无Softmax）
    结构：输入层 → 隐藏层 → 输出层（纯线性变换）
    """
    def __init__(self, input_dim, hidden_dim, output_dim):
        """
        初始化网络参数（权重+偏置）
        :param input_dim: 输入维度（如特征数）
        :param hidden_dim: 隐藏层神经元数
        :param output_dim: 输出维度（如回归任务的1，分类任务的类别数）
        """
        # 输入层→隐藏层的参数
        self.w1 = torch.randn(input_dim, hidden_dim)  # (输入维度, 隐藏层神经元数)
        self.b1 = torch.randn(1, hidden_dim)          # (1, 隐藏层神经元数)
        
        # 隐藏层→输出层的参数
        self.w2 = torch.randn(hidden_dim, output_dim) # (隐藏层神经元数, 输出维度)
        self.b2 = torch.randn(1, output_dim)          # (1, 输出维度)
        
        # 存储前向传播的中间结果（用于反向传播）
        self.x = None       # 输入数据
        self.z1 = None      # 隐藏层线性输出
        self.z2 = None      # 输出层线性输出
        
        # 存储梯度（手写梯度用）
        self.w1_grad = None
        self.b1_grad = None
        self.w2_grad = None
        self.b2_grad = None

    def forward(self, x):
        """
        前向传播（纯线性变换，无激活函数）
        :param x: 输入数据，shape=(样本数N, 输入维度)
        :return: 网络输出，shape=(N, 输出维度)
        """
        self.x = x  # 保存输入，用于反向传播
        
        # 输入层→隐藏层：z1 = x·w1 + b1
        self.z1 = torch.matmul(x, self.w1) + self.b1
        
        # 隐藏层→输出层：z2 = z1·w2 + b2（无激活）
        self.z2 = torch.matmul(self.z1, self.w2) + self.b2
        
        return self.z2

    def compute_loss(self, y_pred, y_true):
        """
        损失函数：均方误差（MSE），适配回归任务（无激活的NN默认做回归）
        :param y_pred: 网络输出，shape=(N, output_dim)
        :param y_true: 真实标签，shape=(N, output_dim)
        :return: 标量损失值
        """
        return torch.mean((y_pred - y_true) ** 2)

    def backward(self, y_true):
        """
        反向传播：手动计算所有参数的梯度（核心！）
        推导逻辑：
        1. 输出层梯度：dL/dz2 = 2*(z2 - y_true)/N
        2. 隐藏层→输出层参数梯度：
           dL/dw2 = z1^T · dL/dz2
           dL/db2 = sum(dL/dz2, dim=0)
        3. 输入层→隐藏层参数梯度：
           dL/dz1 = dL/dz2 · w2^T
           dL/dw1 = x^T · dL/dz1
           dL/db1 = sum(dL/dz1, dim=0)
        """
        N = self.x.shape[0]  # 样本数
        
        # 1. 输出层误差（dL/dz2）
        dz2 = 2 * (self.z2 - y_true) / N  # shape=(N, output_dim)
        
        # 2. 计算w2和b2的梯度
        self.w2_grad = torch.matmul(self.z1.T, dz2)  # (hidden_dim, output_dim)
        self.b2_grad = torch.sum(dz2, dim=0, keepdim=True)  # (1, output_dim)
        
        # 3. 隐藏层误差（dL/dz1）
        dz1 = torch.matmul(dz2, self.w2.T)  # shape=(N, hidden_dim)
        
        # 4. 计算w1和b1的梯度
        self.w1_grad = torch.matmul(self.x.T, dz1)  # (input_dim, hidden_dim)
        self.b1_grad = torch.sum(dz1, dim=0, keepdim=True)  # (1, hidden_dim)

    def step(self, lr):
        """梯度下降更新参数"""
        # 更新隐藏层→输出层参数
        self.w2 -= lr * self.w2_grad
        self.b2 -= lr * self.b2_grad
        
        # 更新输入层→隐藏层参数
        self.w1 -= lr * self.w1_grad
        self.b1 -= lr * self.b1_grad

    def zero_grad(self):
        """清空梯度（避免累积）"""
        self.w1_grad = None
        self.b1_grad = None
        self.w2_grad = None
        self.b2_grad = None

## 构造一个链表/一棵二叉树

In [1]:
import collections
class ListNode:
    def __init__(self,val = 0):
        self.val = val
        self.next = None
    def build_linked_list(self,nums):
        if not nums or nums[0] == 'dull':
            return None
        head = ListNode(nums[0])
        cur = head
        for num in nums[1:]:
            if num != 'dull':
                cur.next = ListNode(num)
                cur = cur.next
        return head
# 测试构建链表
nums = [1, 2, 'dull', 3, 4]
node = ListNode()
head = node.build_linked_list(nums)

# 遍历输出链表值（应输出 1 -> 2）
current = head
while current:
    print(current.val, end=' -> ')
    current = current.next
# 输出：1 -> 2 -> 
import collections
class Tree:
    def __init__(self,val = 0):
        self.val = val
        self.right = None
        self.left = None
    def build_tree(self,nums):
        if not nums or nums[0]=='dull':
            return None
        root = Tree(int(nums[0]))
        queue = collections.deque([root])
        i = 1
        n = len(nums)
        while i<n and queue:
            node = queue.popleft()
            if i<n and nums[i]!='dull':
                node.left = Tree(int(nums[i]))
                queue.append(node.left)
                
            i+=1
            if i<n and nums[i]!='dull':
                node.right = Tree(int(nums[i]))
                queue.append(node.right)
               
            i+=1
        return root
t = Tree()

# 2. 待构建的二叉树数值列表（'dull'表示空节点）
# 列表含义：根节点1 → 左子节点2、右子节点3 → 2的左空（dull）、右4 → 3的左右均空
nums = [1, 2, 3, 'dull', 4]

# 3. 构建二叉树
root = t.build_tree(nums)

# 4. 验证：层序遍历输出二叉树（预期输出：1 2 3 4）
def level_order_traversal(root_node):
    if not root_node:
        return []
    res = []
    q = collections.deque([root_node])
    while q:
        node = q.popleft()
        res.append(node.val)
        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
    return res

# 5. 执行遍历并打印结果
traversal_result = level_order_traversal(root)
print("二叉树层序遍历结果：", traversal_result)
print("预期结果：", [1, 2, 3, 4])


1 -> 2 -> 3 -> 4 -> 二叉树层序遍历结果： [1, 2, 3, 4]
预期结果： [1, 2, 3, 4]


# ML&DL手撕题 （torch实现）

## Loss模块
softmax CE BCE
focalloss
hingeloss(SVM) tripletloss(锚点对比损失) infonceloss（对比损失）
BPRloss mse contrastiveloss
rqvae两部分loss recon+quantize

### softmax CE BCE
BCE（二元交叉熵）
配合Sigmoid激活函数。
CE（多元交叉熵）
配合Softmax激活函数。

In [4]:
#softmax函数和交叉熵损失
import torch
x = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(x.max(dim=1))#返回的是value和index但是没有保留维度
print(x.max(dim=1, keepdim=True))#返回的是value和index并且保留维度
print(x.max(dim=1, keepdim=True)[0])#返回的是value并且保留维度
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))
print(sigmoid(x))
def softmax(x):
    e = torch.exp(x - x.max(dim=1, keepdim=True)[0])#数值稳定的softmax
    return e / e.sum(dim=1, keepdim=True)
print(softmax(x))
def CE(pred, target):
    log_prob = torch.log(softmax(pred))#对预测值进行softmax并取对数
    loss = -log_prob[range(len(target)), target].mean()#计算交叉熵损失
    return loss
pred = torch.tensor([[2.0, 1.0, 0.1],[0.5, 1.5, 2.5]])
target = torch.tensor([0, 2])
print(CE(pred, target))
def BCE(pred, target):
    loss = -(target * torch.log(pred + 1e-15) + (1 - target) * torch.log(1 - pred + 1e-15))
    return loss
pred = torch.tensor([0.2, 0.8])
target = torch.tensor([0, 1])
print(BCE(pred, target))

torch.return_types.max(
values=tensor([3., 6.]),
indices=tensor([2, 2]))
torch.return_types.max(
values=tensor([[3.],
        [6.]]),
indices=tensor([[2],
        [2]]))
tensor([[3.],
        [6.]])
tensor([[0.7311, 0.8808, 0.9526],
        [0.9820, 0.9933, 0.9975]])
tensor([[0.0900, 0.2447, 0.6652],
        [0.0900, 0.2447, 0.6652]])
tensor(0.4123)
tensor([0.2231, 0.2231])


### Focal loss

In [ ]:
# Focal Loss是一种改进的交叉熵损失函数，主要用于处理类别不平衡，对难分类的样本给予更大的权重。
# alpha是平衡因子，alpha较大时模型关注更少数样本较小时则相反。
# gamma是调节因子，gamma越大，易分类样本的损失越小，难分类样本的损失越大。
import torch
import torch.nn.functional as F
def focalloss_multiclass(pred, target, alpha=1.0, gamma=0):
    p_t = F.softmax(pred, dim=-1)
    p_t = p_t[range(len(pred)), target]
    ce_loss = -torch.log(p_t + 1e-15)
    focal_loss = alpha * (1 - p_t) ** gamma * ce_loss
    return focal_loss.mean()
pred = torch.tensor([[0.5, 1.0, 1.5], [2.0, 2.5, 4.0]])  # logits
target = torch.tensor([0, 2])
print(focalloss_multiclass(pred, target))

tensor(0.9933)


### 对比学习模块pointwise|pairwise|listwise|是RS中常见的任务lossfunction设计思路   Hingeloss Tripletloss InfoNCEloss
Pointwise Loss（点对损失）：对每个样本单独建模，常用回归或分类损失（如二分类交叉熵、均方误差）。
Pairwise Loss（对对损失）：对每个正负样本对建模，常用BPR Loss、Hinge Loss等。
 Listwise Loss（列表损失）：对一组样本（一个列表）整体建模，常用Softmax Cross Entropy。

In [ ]:
#Hinge Loss主要用于支持向量机（SVM）等分类模型，旨在最大化类别之间的间隔。
#对于正确分类的样本，损失为0；对于错误分类的样本，损失与预测值和真实标签之间的距离成正比。
#https://en.wikipedia.org/wiki/Hinge_loss
import torch
def hinge_loss(pred, target):
    loss = torch.clamp(1 - pred * target, min=0)
    return loss.mean()
pred = torch.tensor([0.5, -1.0, 1.5])#也就是y
target = torch.tensor([1, -1, 1])
print(hinge_loss(pred, target))
#Triplet Loss目的是把正样本之间拉近，把负样本之间推远。
#三个部分组成：anchor（锚点），positive（正样本）和negative（负样本）
#损失函数的目标是使得anchor与positive之间的距离小于anchor与negative之间的距离至少一个margin。
import torch
def triplet_loss(anchor, positive, negative, margin=1.0):
    pos_dist = torch.norm(anchor - positive, dim=1)
    neg_dist = torch.norm(anchor - negative, dim=1)
    loss = torch.relu(pos_dist - neg_dist + margin)
    return loss.mean()
positive = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
negative = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
anchor = torch.tensor([[1.5, 2.5], [3.5, 4.5]])
print(triplet_loss(anchor, positive, negative))
#INFONCE Loss 一般用于自监督学习（问：啥是自监督，有监督，无监督，半自监督o_O）
import torch
import torch.nn.functional as F
#一般是batch内的正样本和负样本,或者分成锚点和正负样本，如下所示
def info_nce_loss(query, pos, neg, t=0.07):
    """
    query: [batch_size, feature_dim]
    pos: [batch_size, feature_dim]
    neg: [batch_size, num_neg, feature_dim]
    t: 温度参数
    """
    # 归一化
    query = F.normalize(query, dim=-1)  # [B, D]
    pos = F.normalize(pos, dim=-1)      # [B, D]
    neg = F.normalize(neg, dim=-1)      # [B, N, D]
    # 计算相似度

    pos_sim = torch.sum(query * pos, dim=-1,keepdim = True)/t  # [B, 1]
    neg_sim = torch.matmul(query.unsqueeze(1), neg.transpose(1, 2)).squeeze(1)/t
    # 拼接正负样本
    logits = torch.cat([pos_sim, neg_sim], dim=1) # [B, 1+N]
    pos_exp = torch.exp(pos_sim)
    sum_exp = torch.exp(logits).sum(dim=1, keepdim=True)  # [B, 1]
    loss = -torch.log(pos_exp / sum_exp)
    return loss.mean()
query = torch.randn(4, 128)  # [B, D]
pos = torch.randn(4, 128)    # [B, D]
neg = torch.randn(4, 10, 128) # [B, N, D]
print(info_nce_loss(query, pos, neg))

tensor(0.1667)
tensor(0.)
tensor([[5.5228],
        [2.7679],
        [3.3137],
        [5.3177]])
tensor(4.2305)


### BPR loss RS必学！mse loss Contrastive loss

In [ ]:
#BPRloss （贝叶斯个性化排序损失）主要用于RS中的ranking，目的是最大化正样本和负样本之间的差异
#感觉问的确实比较多
import torch as t
torch.manual_seed(42)
#随机生成 3 个用户，每个用户对 6 个项目的评分向量（6维）
User = t.randn(3, 6, 6)
Poss = t.randn(3, 6, 6)
Neg = t.randn(3, 6, 6)
def inter(user,item):
    return (user * item).sum(dim=-1)
def pairloss(user,pos,neg):
    return -t.log(t.sigmoid(inter(user,pos) - inter(user,neg))).m
print(pairloss(User, Poss, Neg))
#MSE loss
import torch
def mse_loss(pred, target):
    return ((pred - target) ** 2).mean()
pred = torch.tensor([0.5, 0.8, 0.3])
target = torch.tensor([0.5, 0.7, 0.2])
print(mse_loss(pred, target))
#Contrastive loss
def contrastive_loss(x1, x2, label, m):
    dist = torch.norm(x1 - x2, p=2, dim=1)#实际上默认是欧式距离
    loss = label * dist**2 + (1 - label) * torch.clamp(m - dist, min=0)**2
    return loss.mean()
x1 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
x2 = torch.tensor([[1.5, 2.5], [5.0, 6.0]])
label = torch.tensor([1, 0])  # 1表示相似，0表示不相似
margin = 1.0
print(contrastive_loss(x1, x2, label, margin))


tensor([[8.6843e-01, 4.3933e+00, 1.2847e-01, 4.6359e-04, 4.8395e-02, 7.9299e-03],
        [1.0685e-01, 1.9286e-02, 3.3854e-02, 9.8161e-03, 1.0031e+00, 7.6724e+00],
        [1.1785e+00, 3.5730e-02, 1.9587e+00, 1.3679e-01, 4.4216e-02, 2.8605e-03]])
tensor(0.0067)
tensor(0.2500)


In [73]:
#我的项目中的RQVAEloss包括有重构损失，协同过滤损失，两部分的码本损失，同时对码本进行了一投影，这部分明天可以详细写一下。

## DL模块
dropout relu swish LN BN RMSNorm MHA mask 各种优化器 二维卷积 MLP

### Dropout relu swish LN BN RMSNorm（只除var不减mean）

In [ ]:
#DL模块
#用torch 
import torch
def dropout(x,p):
    mask = (torch.rand_like(x) > p).float()#float()是为了把bool类型转换成float类型，True变成1.0，False变成0.0😋
    return x * mask / (1 - p)
x = torch.tensor([[1.0, 2.0, 3.0],[4.0, 5.0, 6.0]])
print((torch.rand_like(x)>0.5).float())
print(dropout(x, 0.5))
def swish(x):
    return x * (1 / (1 + torch.exp(-x)))
def relu(x):
    return torch.clamp(x, min=0.0)
print(relu(torch.tensor([-1.0, 0.0, 1.0])))
#LN BN alpha gamma均是可学习参数
#https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html
def LN(x, gamma, alpha, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    #unbiased=True（默认）：计算无偏方差，即分母用 N-1；unbiased=False：计算有偏方差，即分母用N。
    normalized = (x - mean) / torch.sqrt(var + eps)
    return gamma * normalized + alpha
def BN(x, gamma, alpha, eps=1e-5):
    mean = x.mean(dim=0, keepdim=True)
    var = x.var(dim=0, keepdim=True, unbiased=False)
    normalized = (x - mean) / torch.sqrt(var + eps)
    return gamma * normalized + alpha
x = torch.tensor([[1.0, 2.0, 3.0],[4.0, 5.0, 6.0]])
gamma = 1
alpha = 0
print(LN(x, gamma, alpha))
print(BN(x, gamma, alpha))
def RMSNorm(alpha,beta,x):
    var = x.var(dim = -1, keepdim=True, unbiased=False)
    return alpha * x / torch.sqrt(var + 1e-8) + beta

tensor([[0., 0., 1.],
        [0., 1., 1.]])
tensor([[ 0.,  0.,  6.],
        [ 0.,  0., 12.]])
tensor([0., 0., 1.])
tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])
tensor([[-1.0000, -1.0000, -1.0000],
        [ 1.0000,  1.0000,  1.0000]])


### MHA

In [ ]:
#https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
import torch
import torch.nn as nn
class MultiHeadAttention(nn.Module):
    def __init__(self,dim_in,dim_k,dim_v,num_heads = 8):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.dim_k = dim_k
        self.dim_v = dim_v
        self.W_q = nn.Linear(dim_in, dim_k * num_heads, bias=False)
        self.W_k = nn.Linear(dim_in, dim_k * num_heads, bias=False)
        self.W_v = nn.Linear(dim_in, dim_v * num_heads, bias=False)
        self.W_o = nn.Linear(dim_v * num_heads, dim_in, bias=False)
    def forward(self, x):
        batch_size, seq_len, dim_in = x.size()
        q = self.W_q(x).reshape(batch_size, seq_len, self.num_heads, self.dim_k).transpose(1, 2)
        k = self.W_k(x).reshape(batch_size, seq_len, self.num_heads, self.dim_k).transpose(1, 2)
        v = self.W_v(x).reshape(batch_size, seq_len, self.num_heads, self.dim_v).transpose(1, 2)
        #k的维度是(batch_size, num_heads, seq_len, dim_k)，v的维度是(batch_size, num_heads, seq_len, dim_v)
        #k.transpose(-2, -1)的维度是(batch_size, num_heads, dim_k, seq_len)，这样才能和q进行矩阵乘法
        #scores的维度是(batch_size, num_heads, seq_len, seq_len)
        #attn_weights的维度也是(batch_size, num_heads, seq_len, seq_len)
        #attn_output的维度是(batch_size, num_heads, seq_len, dim_v)
        #最后输出的维度是(batch_size, seq_len, dim_in)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.dim_k ** 0.5)
        attn_weights = torch.softmax(scores, dim=-1)
        attn_output = torch.matmul(attn_weights, v).transpose(1, 2).reshape(batch_size, seq_len, self.dim_v * self.num_heads)
        output = self.W_o(attn_output)
        return output
# 创建一个多头注意力层实例
dim_in = 16  # 输入维度
dim_k = 8    # 键和查询的维度
dim_v = 8    # 值的维度
num_heads = 4  # 头的数量
mha = MultiHeadAttention(dim_in, dim_k, dim_v, num_heads)
# 创建一个假设输入数据，形状 (batch_size, seq_len, dim_in)
batch_size = 2
seq_len = 5
x = torch.randn(batch_size, seq_len, dim_in)
output = mha(x)
print("Output shape:", output.shape)

Output shape: torch.Size([2, 5, 16])


### causalmask因果掩码 问：为什么要掩码 encoder decoder掩码方式区别～
一般来说causal和paddingmask只问过我方式

In [18]:
import torch
#triangular upper Matrix上三角矩阵
#triu上三角 tril下三角
def causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)
    mask[mask == 1] = float('-inf')
    return mask
print(causal_mask(5))

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])


import torch
def dropout(x):
    mask = (torch.rand_like(x)>p).float()
    return x*mask/(1-p)

In [22]:
import torch
input_ids = torch.tensor([
    [1, 2, 3, 0, 0],    # 长度3，后面2个是padding
    [4, 5, 6, 7, 8],    # 长度5，无padding
    [9, 0, 0, 0, 0]     # 长度1，后面4个是padding
])
padding_mask = (input_ids == 0)
print(padding_mask)
# tensor([[False, False, False,  True,  True],
#         [False, False, False, False, False],
#         [False,  True,  True,  True,  True]])
attn_score = torch.randn(3, 5, 5)  # 假设的注意力分数
attn_score = attn_score.masked_fill(padding_mask.unsqueeze(1), float('-inf'))
print(attn_score)

tensor([[False, False, False,  True,  True],
        [False, False, False, False, False],
        [False,  True,  True,  True,  True]])
tensor([[[-0.7622, -1.1906,  0.7756,    -inf,    -inf],
         [-1.8815,  0.5851,  1.5287,    -inf,    -inf],
         [-0.6411,  0.5374,  0.7817,    -inf,    -inf],
         [ 1.6077,  1.0938, -0.1177,    -inf,    -inf],
         [ 0.0358,  1.0366,  0.8817,    -inf,    -inf]],

        [[ 0.0585, -1.6682, -0.0200, -1.2131,  1.1971],
         [-0.7818,  1.1162,  0.9625,  0.2794, -0.5718],
         [ 1.9491, -0.4252, -0.3753,  1.4467, -0.7872],
         [-0.2698,  0.1231,  0.8758,  0.2116,  1.2271],
         [-0.5723, -0.1707, -0.4587, -0.3092, -0.0966]],

        [[ 1.5419,    -inf,    -inf,    -inf,    -inf],
         [ 1.6513,    -inf,    -inf,    -inf,    -inf],
         [ 0.1035,    -inf,    -inf,    -inf,    -inf],
         [ 0.1864,    -inf,    -inf,    -inf,    -inf],
         [ 0.3892,    -inf,    -inf,    -inf,    -inf]]])


### SGD Adagrad AdamW Adam RMSprop

In [ ]:
#Adam Adaptive Moment estimation 自适应优化器
#mt = beta1 * m(t-1) + (1 - beta1) * gt
#vt = beta2 * v(t-1) + (1 - beta2) * gt^2
#修偏： mt_hat = mt / (1 - beta1^t) vt_hat = vt / (1 - beta2^t)
#https://arxiv.org/pdf/1412.6980.pdf
import torch
class AdamOptimizer:
    def __init__(self,params,lr = 0.001,b1 = 0.9,b2 = 0.999,eps = 1e-8):
        self.params = list(params)
        self.lr = lr
        self.b1 = b1
        self.b2 = b2
        self.eps = eps
        
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        
        self.t = 0
    def step(self):
        self.t+=1
        for i,param in enumerate(self.params):
            grad = param.grad#获取当前的梯度
            self.m[i] = self.b1*self.m[i] + (1-self.b1)*grad
            self.v[i] = self.b2*self.v[i] + (1-self.b2)*grad**2
            
            m_hat = self.m[i]/(1-self.b1**self.t)
            v_hat = self.v[i]/(1-self.b2**self.t)
            
            param.data -= self.lr*m_hat/(torch.sqrt(v_hat)+self.eps)
    def zero_grad(self):
        for param in self.params:
            param.grad = None
torch.manual_seed(42)
# 创建两个可训练的张量，表示模型的参数
params = [torch.randn(3, 3, requires_grad=True),
          torch.randn(3, 1, requires_grad=True)]
optimizer = AdamOptimizer(params, lr=0.001)
loss = params[0].sum() + params[1].sum()  # 一个简单的损失函数，求参数的和
loss.backward()
optimizer.step()
optimizer.zero_grad()

tensor([[ 0.3367,  0.1288,  0.2345],
        [ 0.2303, -1.1229, -0.1863],
        [ 2.2082, -0.6380,  0.4617]], requires_grad=True)
tensor([[0.2674],
        [0.5349],
        [0.8094]], requires_grad=True)
tensor([[ 0.3267,  0.1188,  0.2245],
        [ 0.2203, -1.1329, -0.1963],
        [ 2.1982, -0.6480,  0.4517]], requires_grad=True)
tensor([[0.2574],
        [0.5249],
        [0.7994]], requires_grad=True)


In [ ]:
import torch

class AdamWOptimizer:
    def __init__(self, params, lr=0.001, b1=0.9, b2=0.999, eps=1e-8, weight_decay=0.01):
        self.params = list(params)
        self.lr = lr
        self.b1 = b1
        self.b2 = b2
        self.eps = eps
        self.weight_decay = weight_decay
        
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        
        self.t = 0

    def step(self):
        self.t += 1
        for i, param in enumerate(self.params):
            grad = param.grad
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * grad
            self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * grad ** 2

            m_hat = self.m[i] / (1 - self.b1 ** self.t)
            v_hat = self.v[i] / (1 - self.b2 ** self.t)
            param.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps) + self.lr * self.weight_decay * param.data
    def zero_grad(self):
        for param in self.params:
            param.grad = None

torch.manual_seed(42)
params = [torch.randn(3, 3, requires_grad=True),
          torch.randn(3, 1, requires_grad=True)]
optimizer = AdamWOptimizer(params, lr=0.001, weight_decay=0.01)

loss = params[0].sum() + params[1].sum()
loss.backward()

optimizer.step()
optimizer.zero_grad()

# 查看更新后的参数
print(params[0])
print(params[1])

tensor([[ 0.3357,  0.1278,  0.2335],
        [ 0.2293, -1.1238, -0.1873],
        [ 2.2072, -0.6390,  0.4607]], requires_grad=True)
tensor([[0.2663],
        [0.5339],
        [0.8083]], requires_grad=True)
tensor([[ 1.1103, -1.6898, -0.9890],
        [ 0.9580,  1.3221,  0.8172]])
tensor([[1, 0],
        [0, 0]])


In [ ]:
import torch
class SGDOptimizer:
    def __init__(self,params,lr = 0.01):
        self.params = list(params)
        self.lr = lr
    def step(self):
        for param in self.params:
            param.data-=self.lr*param.grad
    def zero_grad(self):
        for param in self.params:
            param.grad = None
torch.manual_seed(42)
params = [torch.randn(3, 3, requires_grad=True),
            torch.randn(3, 1, requires_grad=True)]  
optimizer = SGDOptimizer(params, lr=0.01)
loss = params[0].sum() + params[1].sum()
loss.backward()
optimizer.step()
optimizer.zero_grad()
print(params[0])
print(params[1])

tensor([[ 0.3267,  0.1188,  0.2245],
        [ 0.2203, -1.1329, -0.1963],
        [ 2.1982, -0.6480,  0.4517]], requires_grad=True)
tensor([[0.2574],
        [0.5249],
        [0.7994]], requires_grad=True)
tensor([[ 0.3267,  0.1188,  0.2245],
        [ 0.2203, -1.1329, -0.1963],
        [ 2.1982, -0.6480,  0.4517]], requires_grad=True)
tensor([[0.2574],
        [0.5249],
        [0.7994]], requires_grad=True)


In [71]:
#写多了就会发现 SGD Adagrad AdamW Adam RMSprop都大同小异
import torch
class Adagrad:
    def __init__(self,params,lr,eps=1e-10):
        self.params = list(params)
        self.lr = lr
        self.eps = eps
        self.v = [torch.zeros_like(p) for p in self.params]
    def step(self):
        for i,param in enumerate(self.params):
            grad = param.grad
            self.v[i]+= grad**2
            param.data -= self.lr*grad/(torch.sqrt(self.v[i])+1e-9)
    def zero_grad(self):
        for param in self.params:
            param.grad = None
torch.manual_seed(42)
params = [torch.randn(3, 3, requires_grad=True),
          torch.randn(3, 1, requires_grad=True)]
optimizer = Adagrad(params, lr=0.01)
loss = params[0].sum() + params[1].sum()
loss.backward()
optimizer.step()
optimizer.zero_grad()
print(params[0])
print(params[1])

tensor([[ 0.3267,  0.1188,  0.2245],
        [ 0.2203, -1.1329, -0.1963],
        [ 2.1982, -0.6480,  0.4517]], requires_grad=True)
tensor([[0.2574],
        [0.5249],
        [0.7994]], requires_grad=True)


### 2维卷积

In [ ]:
import torch
def conv2d(input, kernel):
    H,W = input.shape
    kH,kW = kernel.shape
    out_H = H - kH + 1
    out_W = W - kW + 1
    output = torch.zeros(out_H, out_W)
    for i in range(out_H):
        for j in range(out_W):
            output[i, j] = (input[i:i+kH, j:j+kW] * kernel).sum()
    return output
input = torch.tensor([[1.0, 2.0, 3.0],
                      [4.0, 5.0, 6.0],
                      [7.0, 8.0, 9.0]])
kernel = torch.tensor([[0.0, 1.0],
                       [1.0, 0.0]])
print(conv2d(input, kernel))

import torch
def conv2d(input,kernel):
    H,W = input.shape
    kh,kw = kernel.shape
    out_H = H-kh+1
    out_W = W-kw+1
    output = torch.zeros(out_H, out_W)
    for i in range(out_H):
        for j in range(out_W):
            output[i,j] = (input[i:i+kh,j:j+kw]*kernel).sum()
    return output

tensor([[ 6.,  8.],
        [12., 14.]])


### MLP（CE+backward+relu+softmax）

In [ ]:
import torch
def softmax(x):
    e = torch.exp(x-x.max(dim = 1,keepdim = True)[0])
    return e/e.sum(dim = 1,keepdim = True)
def relu(x):
    return torch.maximum(x,torch.zeros_like(x))
def relu_derivative(x):
    return (x>0).float()
class MLP:
    def __init__(self,input_sizes,hidden_sizes,output_sizes):
        self.W1 = torch.randn(input_sizes,hidden_sizes)
        self.b1 = torch.randn(1,hidden_sizes)
        self.W2 = torch.randn(hidden_sizes,output_sizes)
        self.b2 = torch.randn(1,output_sizes)

        self.cache = {}

    def forward(self,X):
        self.cache["X"] = X
        self.cache["Z1"] = torch.matmul(X,self.W1)+self.b1#B,H
        self.cache["A1"] = relu(self.cache["Z1"])#B,H

        self.cache["Z2"] = torch.matmul(self.cache["A1"],self.W2)+self.b2#B,O
        self.cache["A2"] = softmax(self.cache["Z2"])
        return self.cache["A2"]
    def backward(self,Y_true,lr = 0.01):
        m = Y_true.shape[0]
        Y_pred = self.cache["A2"]
        dZ2 = Y_pred - Y_true#B,O
        dW2 = torch.matmul(self.cache["A1"].T,dZ2)/m#H,O
        db2 = torch.sum(dZ2,dim = 0,keepdim=True)/m#1,O

        dA1 = torch.matmul(dZ2,self.W2.T)
        dZ1 = dA1*relu_derivative(self.cache["Z1"])

        dW1 = torch.matmul(self.cache["X"].T,dZ1)/m
        db1 = torch.sum(dZ1,dim = 0,keepdim = True)/m
        
        self.W1 -= lr*dW1
        self.b1 -= lr*db1
        self.W2 -= lr*dW2
        self.b2 -= lr*db2
    def ce(self,y_true,y_pred):
        y_pred = torch.clamp(y_pred,1e-8,1-1e-8)
        loss = -torch.log(y_pred[range(len(y_true)),y_true]).mean()
        return loss
if __name__ == "__main__":
    # 1. 准备数据
    X = torch.tensor([[1.0, 2.0], [2.0, 3.0], [3.0, 4.0]], dtype=torch.float32)
    Y_indices = torch.tensor([0, 1, 2], dtype=torch.long)  # 用于 CE 损失
    
    # 构造 One-Hot 用于 backward (因为 backward 写的是 Y_pred - Y_true)
    Y_onehot = torch.zeros(3, 3)
    Y_onehot[range(3), Y_indices] = 1.0
    
    # 2. 初始化模型
    model = MLP(input_sizes=2, hidden_sizes=4, output_sizes=3)
    
    # 3. 前向传播
    pred = model.forward(X)
    print("初始损失:", model.ce(Y_indices, pred))
    
    # 4. 反向传播
    model.backward(Y_onehot, lr=0.1)
    
    # 5. 再次前向
    pred = model.forward(X)
    print("更新后损失:", model.ce(Y_indices, pred))


初始损失: tensor(3.2574)
更新后损失: tensor(1.8725)


In [ ]:
def softmax(x):
    e = torch.exp(x - x.max(dim = 1,keepdim = True)[0])
    return e/e.sum(dim = 1,keepdim = True)
def relu(x):
    return torch.maximum(x,torch.zeros_like(x))


## ML 模块
kmeans kmeans++ LR LR

### Kmeans and Kmeans_plus

In [207]:
import torch
def kmeans(x,k,iter = 100):
    torch.manual_seed(42)
    n = x.shape[0]
    c = x[torch.randperm(n)[:k]]#生成一个0到n-1的随机排列，取前k个作为初始中心
    for i in range(iter):
        d = torch.cdist(x,c)#计算每个点到每个中心的距离
        l = torch.argmin(d,dim=1)#每个点分配到最近的中心
        c_ = torch.stack([x[l==j].mean(0) for j in range(k)])
        if torch.norm(c-c_) < 1e-5:
            break
        c = c_
    return c,l
x = torch.randn(100,2)
c,l = kmeans(x,10)
print(c)
def kmeansplus(x,k,iter = 100):
    torch.manual_seed(42)
    n = x.shape[0]
    first_idx = torch.randint(n,(1,))#随机选择第一个中心
    c = [x[first_idx]]
    for _ in range(1,k):
        c_tensor = torch.cat(c, dim=0)

        d = torch.cdist(x,c_tensor)#计算每个点到当前中心的距离
        p = d.min(1)[0]**2#距离的平方作为概率
        p /= p.sum()#归一化概率
        
        c.append(x[torch.multinomial(p, 1)])#根据概率选择下一个中心
    c = torch.cat(c,dim = 0)
    for i in range(iter):
        d = torch.cdist(x,c)
        l = torch.argmin(d,dim=1)
        c_ = torch.stack([x[l==j].mean(0) for j in range(k)])
        if torch.norm(c-c_) < 1e-5:
            break
        c = c_
    return c,l
x = torch.randn(100,2)
c,l = kmeansplus(x,10)
print(c)

tensor([[ 0.1024, -1.4960],
        [-1.3257, -0.2296],
        [ 0.8559, -0.4050],
        [-0.7327,  1.1278],
        [ 1.4961,  0.4438],
        [ 1.2632,  3.3657],
        [ 1.4660, -1.3017],
        [-0.2501,  0.4768],
        [ 0.3207,  1.1832],
        [-0.4002, -0.7275]])
tensor([[-0.6904,  0.9859],
        [ 0.6098,  0.7237],
        [-1.6535, -0.7003],
        [ 0.6856, -0.4346],
        [-0.6049, -0.0294],
        [ 0.9009, -1.8033],
        [-2.3184,  1.3032],
        [-0.2908, -1.2381],
        [ 1.6817,  0.3484],
        [ 0.0549,  0.5853]])


In [9]:
import torch
import random
def kmeans(x,k,iter):
    torch.manual_seed(42)
    n = x.shape[0]
    c = x[torch.randperm(n)[:k]]
    for _ in range(iter):
        d = torch.cdist(x,c)
        l = torch.argmin(d,dim = 1)
        c_= torch.stack([x[l==j].mean(0) for j in range(k)])
        if torch.norm(c-c_)<1e-5:
            break
        c = c_
    return c,l
x = torch.randn(100,2)
c,l = kmeans(x,10,100)
print(c)
def kmeansplus(x,k,iter):
    torch.manual_seed(42)
    n = x.shape[0]
    first_idx = torch.randint(n,(1,))
    c = [x[first_idx]]
    for i in range(1,k):
        c_ = torch.cat(c,dim = 0)
        
        d = torch.cdist(x,c_)
        p = d.min(1)[0]
        p = p/p.sum()

        c.append(x[torch.multinomial(p,1)])
    c = torch.cat(c,dim = 0)
    for i in range(iter):
        d = torch.cdist(x,c)
        l = torch.argmin(d,dim=1)
        c_ = torch.stack([x[l==j].mean(0) for j in range(k)])
        if torch.norm(c-c_) < 1e-5:
            break
        c = c_
    return c,l
x = torch.randn(100,2)
c,l = kmeansplus(x,10,100)
print(c)

tensor([[ 0.9996, -0.3402],
        [-1.0685,  1.2570],
        [ 2.0460,  1.1660],
        [-1.0061,  0.1026],
        [ 0.3366,  0.5218],
        [-0.1523, -0.2058],
        [ 0.9339, -1.9726],
        [ 1.3619,  0.5334],
        [-0.0348,  1.3756],
        [-0.2206, -1.2397]])
tensor([[-0.6904,  0.9859],
        [ 1.6613,  0.6401],
        [-1.6535, -0.7003],
        [ 0.9874, -0.4305],
        [-0.5796,  0.1386],
        [ 0.1729, -2.1993],
        [-2.3184,  1.3032],
        [ 0.2205, -1.3318],
        [-0.3740, -0.8144],
        [ 0.3563,  0.6579]])


### linear regression & logistic regression

In [ ]:
import torch
class linear_regression:
    def __init__(self,input_size):
        self.w = torch.randn(input_size,1)
        self.b = torch.randn(1,1)
    def forward(self,x):
        self.y_pred = torch.matmul(x,self.w)+self.b
        return self.y_pred
    def backward(self,x,y_true):
        N = x.shape[0]
        error = self.y_pred - y_true
        self.w_grad = (2/N)*torch.matmul(x.T,error)
        self.b_grad= (2/N)*torch.sum(error)
    def step(self,lr):
        self.w -= lr * self.w_grad
        self.b -= lr * self.b_grad
    def zero_grad(self):
        self.w_grad = None
        self.b_grad = None
x = torch.randn(100,10)
y = torch.randint(2,(100,)).unsqueeze(1)
n = x.shape[1]
model = linear_regression(n)
for i in range(10):
    model.zero_grad()
    model.forward(x)
    model.backward(x,y)
    model.step(0.01)
    loss = torch.mean((model.y_pred - y) ** 2)
    print(f"第{i+1}轮，MSE损失：{loss.item():.4f}")
import torch
def sigmoid(x):
    return 1/(1+torch.exp(-x))
class logistic_regression:
    def __init__(self,input_size):
        self.w = torch.randn(input_size,1)
        self.b = torch.randn(1,1)
    def forward(self,x):
        z = torch.matmul(x,self.w)+self.b
        self.y_pred = sigmoid(z)
        return self.y_pred  
    def backward(self,x,y_true):
        N = len(y_true)
        error = self.y_pred - y_true
        self.w_grad = (1/N)*torch.matmul(x.T,error)
        self.b_grad = (1/N)*torch.sum(error)
    def step(self,lr):
        self.w -= lr*self.w_grad
        self.b -= lr*self.b_grad
    def zero_grad(self):
        self.w_grad = None
        self.b_grad = None
x = torch.randn(100,10)
y = torch.randint(2,(100,)).unsqueeze(1)
n = x.shape[1]
model = logistic_regression(n)
for i in range(10):
    model.zero_grad()
    model.forward(x)
    model.backward(x,y)
    model.step(0.01)
    loss = torch.mean((model.y_pred - y) ** 2)
    print(f"第{i+1}轮，MSE损失：{loss.item():.4f}")


第1轮，MSE损失：0.4198
第2轮，MSE损失：0.4197
第3轮，MSE损失：0.4196
第4轮，MSE损失：0.4195
第5轮，MSE损失：0.4195
第6轮，MSE损失：0.4194
第7轮，MSE损失：0.4193
第8轮，MSE损失：0.4192
第9轮，MSE损失：0.4192
第10轮，MSE损失：0.4191


## 模型评价指标
AUC GAUC Accuracy precision recall f1socre topk topp

### AUC｜GAUC 

In [204]:
def auc_score(y_true, y_pred):
    """
    y_true: 真实标签（0或1），形状 [n]
    y_pred: 预测概率，形状 [n]
    """
    pos = []
    neg = []
    for yt, yp in zip(y_true, y_pred):
        if yt == 1:
            pos.append(yp)
        else:
            neg.append(yp)
    num = 0
    total = 0
    for p in pos:
        for n in neg:
            if p > n:
                num += 1
            elif p == n:
                num += 0.5
            total += 1
    return num / total if total > 0 else 0
y_true = [0, 1, 0, 1, 1, 0]
y_pred = [0.1, 0.8, 0.3, 0.7, 0.6, 0.2]
print("AUC:", auc_score(y_true, y_pred))

from collections import defaultdict
def gauc_score(group_ids,y_true,p_pred):
    group_data = defaultdict(list)
    for gid,yt,yp in zip(group_ids,y_true,y_pred):
        group_data[gid].append((yt,yp))
    total_pairs = 0
    weighted_auc = 0
    for gid,samples in group_data.items():
        y_g = [s[0] for s in samples]
        y_p = [s[1] for s in samples]
        pos_num = sum(y_g)
        neg_num = len(y_g)-pos_num
        pair_num = pos_num*neg_num
        if pair_num == 0:
            continue
        auc = auc_score(y_g,y_p)
        weighted_auc += auc * pair_num
        total_pairs += pair_num
        return weighted_auc / total_pairs if total_pairs > 0 else 0
group_ids = ['A', 'A', 'B', 'B', 'B', 'C']
y_true =    [0,   1,   0,   1,   1,   0]
y_pred =    [0.1, 0.8, 0.3, 0.7, 0.6, 0.2]
print("GAUC:", gauc_score(group_ids, y_true, y_pred))

AUC: 1.0
GAUC: 1.0


### Accuracy|Precision|Recall|F1_score

In [205]:
def accuracy(y_true, y_pred):
    correct = sum([yt == yp for yt, yp in zip(y_true, y_pred)])
    return correct / len(y_true)
def precision(y_true, y_pred):
    tp = sum([yt == 1 and yp == 1 for yt, yp in zip(y_true, y_pred)])
    fp = sum([yt == 0 and yp == 1 for yt, yp in zip(y_true, y_pred)])
    return tp / (tp + fp) if (tp + fp) > 0 else 0
def recall(y_true, y_pred):
    tp = sum([yt == 1 and yp == 1 for yt, yp in zip(y_true, y_pred)])
    fn = sum([yt == 1 and yp == 0 for yt, yp in zip(y_true, y_pred)])
    return tp / (tp + fn) if (tp + fn) > 0 else 0
def f1_score(y_true, y_pred):
    p = precision(y_true, y_pred)
    r = recall(y_true, y_pred)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0
y_true = [0, 1, 1, 0, 1, 0, 1, 1]
y_pred = [0, 1, 0, 0, 1, 1, 1, 1]

print("Accuracy:", accuracy(y_true, y_pred))
print("Precision:", precision(y_true, y_pred))
print("Recall:", recall(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))

Accuracy: 0.75
Precision: 0.8
Recall: 0.8
F1-score: 0.8000000000000002


### Top k / p

In [206]:
def top_k_accuracy(y_true, y_pred_scores, k=5):
    """
    y_true: 真实类别索引（如[2, 0, 1]）
    y_pred_scores: 每个样本的预测分数（如[[0.1,0.7,0.2], ...]）
    k: top-k值
    """
    correct = 0
    for yt, scores in zip(y_true, y_pred_scores):
        topk = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        if yt in topk:
            correct += 1
    return correct / len(y_true)
y_true = [2, 0, 1]
y_pred_scores = [
    [0.1, 0.7, 0.2],  # 真实类别2，分数最高的是1、2、0
    [0.8, 0.1, 0.1],  # 真实类别0，分数最高的是0、1、2
    [0.3, 0.5, 0.2]   # 真实类别1，分数最高的是1、0、2
]
print("Top-2 Accuracy:", top_k_accuracy(y_true, y_pred_scores, k=2))

def top_p_accuracy(y_true, y_pred_scores, p=0.8):
    """
    y_true: 真实类别索引
    y_pred_scores: 每个样本的预测分数（如概率）
    p: top-p阈值
    """
    correct = 0
    for yt, scores in zip(y_true, y_pred_scores):
        # 排序
        sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        sorted_scores = [scores[i] for i in sorted_indices]
        cum_prob = 0
        top_p_indices = []
        for idx, prob in zip(sorted_indices, sorted_scores):
            cum_prob += prob
            top_p_indices.append(idx)
            if cum_prob >= p:
                break
        if yt in top_p_indices:
            correct += 1
    return correct / len(y_true)
y_true = [2, 0, 1]
y_pred_scores = [
    [0.1, 0.7, 0.2],  # 真实类别2
    [0.8, 0.1, 0.1],  # 真实类别0
    [0.3, 0.5, 0.2]   # 真实类别1
]
print("Top-p(0.8) Accuracy:", top_p_accuracy(y_true, y_pred_scores, p=0.8))

Top-2 Accuracy: 1.0
Top-p(0.8) Accuracy: 1.0
